# Front-End Pretraining — Neutral Head  (HPC full-training twin)

Full-resolution (`256^3`) pretraining of the **shared bi-planar front-end** (ConvNeXtV2 encoder +
hybrid fusion + 2D→3D lift) on the Sunway HPC GPU, against the **4-channel per-bone** target. The
SimCLR-pretrained encoder stays **frozen**; only the **fusion + lift** are trained with a throwaway
**neutral SingleConv head**, then the front-end is saved to `models/front_end_fold{FOLD}.pth`.

`03_decoder_pipeline.ipynb` loads that file, **freezes the whole front-end**, and trains each decoder on
top — so U-Net and V-Net consume *byte-identical* features and the only difference is the decoder
block. This is the twin of `notebooks/modeling/02_frontend_pretrain.ipynb`; the **only** differences are
in the CONFIG cell.

Pipeline: `AP+LAT DRRs -> frozen SimCLR encoder -> fusion+lift (TRAIN) -> neutral SingleConv head -> per-bone occupancy volumes`

### How to run

1. Run the cells top to bottom. The **CONFIG** cell is the only place you change settings.
2. This trains the front-end **once per fold** with the neutral head and writes
   `models/front_end_fold{FOLD}.pth` plus the per-fold split `models/decoders/decoder_split_fold{FOLD}.csv`.
3. For the cross-validated comparison, run this notebook for **each `FOLD` in `0..N_FOLDS-1`**, then
   open `03_decoder_pipeline.ipynb` (same `FOLD`, `REGIME="frozen"`) for `MODEL="unet"` and
   `MODEL="vnet"` — both load this fold's frozen front-end, so only the decoder differs.

This notebook requires the encoder checkpoint `models/convnextv2_simclr_encoder.pth` produced by
`01_encoder_pipeline.ipynb`. The front-end is pretrained **per fold** (it uses occupancy labels), so the
U-Net/V-Net comparison stays free of label leakage.

In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")  # reduce CUDA fragmentation (OOM safeguard); must be set before torch initialises CUDA
import math, random, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint as cp
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm
import nibabel as nib
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("torch", torch.__version__, "| timm", timm.__version__, "| cuda:", torch.cuda.is_available())

In [ ]:
# ============================= CONFIG (HPC - full 256^3 front-end pretraining) =============================
# Mirrors the local notebook; only these knobs differ. Run on the Sunway HPC GPU.
ENV          = "HPC"
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL        = "neutral"     # neutral SingleConv head: pretrains the shared front-end, favouring no decoder
TARGET_RES   = 256
LIFT_DEPTH   = 16            # MUST match 03_decoder_pipeline.ipynb
EPOCHS       = 40
BATCH_SIZE   = 1             # 256^3 is memory-heavy; raise to 2 only if the GPU allows
LR           = 1e-4
CKPT_EVERY   = 5
USE_AMP      = True
USE_GRAD_CKPT = True
NUM_WORKERS  = 4
INCLUDE_GEOMETRIC = False    # MUST match 03_decoder_pipeline.ipynb (same train split, no val/test leakage)
DEEP_SUPERVISION  = False    # not used by the neutral head
FREEZE_ENCODER = True        # freeze the pretrained (FCMAE) backbone; train only fusion + 2D->3D lift (+ head)
STEP2_TARGET   = "tsdf_blend" # step-2 objective: "binary" (arm a) | "tsdf_blend" (C-1: 0.5*TSDF-L1 + 0.5*BCE/Dice)
# --- per-bone multi-label target (femur/tibia/patella/fibula), one channel per bone ---
N_CLASSES    = 4
BONES        = ["femur", "tibia", "patella", "fibula"]
# --- cross-validation (knee-level, dataset-stratified) — MUST match 03_decoder_pipeline.ipynb ---
# The front-end uses per-bone GT (labels), so it is pretrained PER FOLD on that fold's train knees
# to keep the U-Net/V-Net comparison free of label leakage. This notebook is the PRODUCER of the
# per-fold split CSV that 03_decoder_pipeline.ipynb then loads.
N_FOLDS      = 5             # k-fold CV over knees (matches FracReconNet); run every fold
FOLD         = 0             # which fold this front-end is for (0..N_FOLDS-1)
PRETRAINED   = True
SMOKE_TEST   = False
SMOKE_CASES_PER_GROUP = 3
RESUME_FROM  = None
EXPLICIT_ROOT = None         # e.g. "/home/project/xray2mesh/Marcus_Chan_Zheng_Shao_CP2 _24020059"
if DEVICE.type != "cuda":
    print("[warning] CUDA not available - this HPC notebook expects a GPU.")
print("ENV", ENV, "| MODEL", MODEL, "| fold", FOLD, "/", N_FOLDS,
      "| TARGET_RES", TARGET_RES, "| device", DEVICE, "| epochs", EPOCHS)

In [ ]:
# Resolve the project root robustly (works locally and on HPC, regardless of where
# the notebook is launched from). We look upward for the data/interim/predrr folder,
# which anchors the project (the per-bone GT lives next to it under gt_per_bone_256).
def find_root(start: Path) -> Path:
    if EXPLICIT_ROOT:
        r = Path(EXPLICIT_ROOT)
        if (r / "data" / "interim" / "predrr").exists():
            return r
    p = start.resolve()
    for cand in [p, *p.parents]:
        if (cand / "data" / "interim" / "predrr").exists():
            return cand
    raise FileNotFoundError("Could not find project root (expected data/interim/predrr). "
                            "Set EXPLICIT_ROOT in the CONFIG cell.")

ROOT           = find_root(Path.cwd())
DATA           = ROOT / "data"
NORMAL_DRR_DIR = DATA / "interim" / "DRRs"                 # AP/LAT DRRs (model inputs)
AUG_DRR_DIR    = DATA / "processed" / "augmented_DRRs"     # augmented DRR variants
PREDRR_DIR     = DATA / "interim" / "predrr"               # bone-windowed CT (root anchor; not the target)
GT_PB_DIR      = DATA / "interim" / "gt_per_bone_256"      # NEW target: per-bone STL-derived GT (femur/tibia/patella/fibula)
MODELS_DIR     = ROOT / "models"
# Encoder init: prefer the label-free FCMAE encoder exported by 01_encoder_pipeline.ipynb;
# fall back to the legacy SimCLR checkpoint only if FCMAE is absent (SimCLR is retired).
FCMAE_CKPT     = MODELS_DIR / "convnextv2_fcmae_encoder.pth"       # label-free FCMAE (+cross-view) encoder
SIMCLR_CKPT    = MODELS_DIR / "convnextv2_simclr_encoder.pth"      # legacy fallback only (retired)
ENCODER_CKPT   = FCMAE_CKPT if FCMAE_CKPT.exists() else SIMCLR_CKPT # weights are fold-agnostic; pretraining uses no labels
FRONTEND_CKPT  = MODELS_DIR / ("front_end_fold%d.pth" % FOLD)      # <- per-fold output; 03_decoder_pipeline.ipynb loads the matching FOLD
CKPT_DIR       = MODELS_DIR / "decoders" / ("neutral_fold%d" % FOLD)   # per-fold neutral-head checkpoints (intermediate)
CKPT_DIR.mkdir(parents=True, exist_ok=True)
GT_CACHE_DIR   = DATA / "interim" / ("gt_per_bone_occ_%d" % TARGET_RES)   # cached (4,T,T,T) per-bone GT at TARGET_RES
GT_TSDF_CACHE_DIR = DATA / "interim" / ("gt_per_bone_tsdf_%d" % TARGET_RES)  # cached (4,T,T,T) TSDF at TARGET_RES (C-1)
print("ROOT:", ROOT)
print("encoder init ->", ENCODER_CKPT.name, "(exists:", ENCODER_CKPT.exists(), ")")
print("front-end ->", FRONTEND_CKPT)
print("checkpoints ->", CKPT_DIR)

## 1. Shared encoder (copied verbatim from `01_encoder_pipeline.ipynb`)

The encoder is the part both decoders share, so the comparison is fair: **only the decoder
changes**. The code below is copied verbatim from `01_encoder_pipeline.ipynb` so this notebook is
self-contained — **do not edit it here**. (When `01_encoder_pipeline.ipynb` is later converted to a
`.py` module, replace this cell with a simple `import`.)

What it does, in plain terms:
1. A **ConvNeXtV2** backbone turns each X-ray (AP and LAT) into 4 feature maps at increasing depth.
2. **Hybrid bi-planar fusion** merges the two views: cheap convolution at fine scales (keeps local
   fracture detail), cross-attention at coarse scales (aligns global knee shape).
3. A **2D->3D lift** stacks each fused map into a small 3D feature volume (depth = `LIFT_DEPTH`).

Output: a list of 4 multi-scale 3D feature tensors with channels `[64, 128, 256, 512]` — this is
the *contract* the decoder consumes.

In [ ]:
# ===== Encoder front-end - VERBATIM from 01_encoder_pipeline.ipynb. DO NOT EDIT. =====
# (PRETRAINED / FREEZE_ENCODER are set in the CONFIG cell so they stay visible knobs.)
BACKBONE     = "convnextv2_tiny"
IMG_SIZE     = 256
OUT_CHANNELS = [64, 128, 256, 512]
FUSION_TYPES = ["local", "local", "attn", "attn"]   # fine -> coarse

def make_backbone(pretrained=True):
    """features_only ConvNeXtV2 returning 4 multi-scale maps. Falls back to random init offline."""
    try:
        return timm.create_model(BACKBONE, pretrained=pretrained, features_only=True)
    except Exception as e:
        print("[warn] pretrained fetch failed (%s); random init." % type(e).__name__)
        return timm.create_model(BACKBONE, pretrained=False, features_only=True)

FEAT_DIMS = [f["num_chs"] for f in make_backbone(pretrained=False).feature_info]   # [96,192,384,768]

def load_drr(path):
    """npy 256x256 float32 [0,1] -> tensor [3,H,W] (1 channel replicated to 3 for ConvNeXtV2)."""
    arr = np.load(path).astype(np.float32)
    t = torch.from_numpy(arr)
    if t.ndim == 2:
        t = t.unsqueeze(0)
    return t.repeat(3, 1, 1) if t.shape[0] == 1 else t

NORMALIZE = T.Normalize(mean=[0.5] * 3, std=[0.5] * 3)
def paired_tf(t):
    return NORMALIZE(t)

class CrossAttention(nn.Module):
    """AP (query) attends to LAT (key/value). Operates on tokens [B, N, C]."""
    def __init__(self, dim):
        super().__init__()
        self.q = nn.Linear(dim, dim); self.k = nn.Linear(dim, dim); self.v = nn.Linear(dim, dim)
        self.scale = dim ** -0.5
    def forward(self, a, b):
        attn = F.softmax(torch.matmul(self.q(a), self.k(b).transpose(-2, -1)) * self.scale, dim=-1)
        return torch.matmul(attn, self.v(b)) + a

class LocalFusion(nn.Module):
    """Cheap high-res fusion: concat views + 3x3 conv, residual on AP."""
    def __init__(self, dim):
        super().__init__()
        self.mix = nn.Conv2d(2 * dim, dim, kernel_size=3, padding=1)
    def forward(self, a, b):
        return self.mix(torch.cat([a, b], dim=1)) + a

# --- bi-planar lift orientation (resolved empirically; see Check 1 / _axis_probe) ---
# GT array axes, from nibabel axcodes ('L','P','S'): axis0 = L-R, axis1 = A-P, axis2 = S-I.
# AP projects along A-P (axis1); LAT projects along L-R (axis0). Both DRR rows (H) = S-I (axis2);
# AP cols (W) = L-R (axis0); LAT cols (W) = A-P (axis1). The fused cube is ordered (axis0, axis1,
# axis2) to match the GT array. flip_* reverse a row/col direction vs its volume axis; locked from
# the affine + 1-D S-I profile test and re-confirmed by the one-sample overfit guard.
LIFT_FLIP_SI      = False   # DRR rows  vs axis2 (S-I)
LIFT_FLIP_AP_COL  = False   # AP  cols  vs axis0 (L-R)
LIFT_FLIP_LAT_COL = True    # LAT cols  vs axis1 (A-P)

class BiPlanarFeatureFusion(nn.Module):
    def __init__(self, feat_dims=FEAT_DIMS, out_channels=OUT_CHANNELS,
                 fusion_types=FUSION_TYPES, depth=16, pretrained=True, freeze_encoder=False):
        super().__init__()
        self.encoder = make_backbone(pretrained)
        self.fusion_types = list(fusion_types)
        self.depth = depth   # retained for signature compat; the orthogonal lift no longer uses it
        self.fuse = nn.ModuleList([CrossAttention(d) if t == "attn" else LocalFusion(d)
                                   for d, t in zip(feat_dims, fusion_types)])
        self.to3d = nn.ModuleList([nn.Conv2d(c, o, 1) for c, o in zip(feat_dims, out_channels)])
        # expand3d fuses the two orthogonally back-projected view cubes (2*o -> o) in 3D
        self.expand3d = nn.ModuleList([nn.Conv3d(2 * o, o, 3, padding=1) for o in out_channels])
        if freeze_encoder:
            for p in self.encoder.parameters():
                p.requires_grad = False
    def load_simclr_encoder(self, path):
        missing, unexpected = self.encoder.load_state_dict(torch.load(path, map_location="cpu"), strict=False)
        print("loaded SimCLR encoder: missing=%d unexpected=%d" % (len(missing), len(unexpected)))
    def _ortho_lift(self, ap_f, lat_f, c2d, c3d):
        """Orthogonal back-projection lift. Each view is placed on the two volume axes it resolves
        and broadcast along its (unobserved) projection axis; the two view cubes are then fused in
        3D. Preserves bi-planar depth instead of extruding one fused 2D map along Z. Output cube is
        ordered (axis0=L-R, axis1=A-P, axis2=S-I) to match the GT array."""
        B, C, H, W = ap_f.shape           # square feature map at this level: S = H = W
        S = H
        ap = c2d(ap_f); lat = c2d(lat_f)  # shared 1x1 projection -> [B, O, H, W] each
        O = ap.shape[1]
        if LIFT_FLIP_SI:      ap = ap.flip(2); lat = lat.flip(2)   # rows (H) = S-I (axis2)
        if LIFT_FLIP_AP_COL:  ap = ap.flip(3)                       # AP  cols (W) = L-R (axis0)
        if LIFT_FLIP_LAT_COL: lat = lat.flip(3)                     # LAT cols (W) = A-P (axis1)
        # AP: (H=axis2, W=axis0) -> cube (axis0, axis1, axis2), broadcast over axis1 (A-P)
        ap_cube = ap.permute(0, 1, 3, 2).unsqueeze(3).expand(B, O, S, S, S)
        # LAT: (H=axis2, W=axis1) -> cube (axis0, axis1, axis2), broadcast over axis0 (L-R)
        lat_cube = lat.permute(0, 1, 3, 2).unsqueeze(2).expand(B, O, S, S, S)
        return c3d(torch.cat([ap_cube, lat_cube], dim=1))           # [B, O, S, S, S]
    def forward(self, ap_img, lat_img):
        ap_feats, lat_feats = self.encoder(ap_img), self.encoder(lat_img)
        fused2d, fused3d = [], []
        for ap_f, lat_f, fuse, c2d, c3d, t in zip(
                ap_feats, lat_feats, self.fuse, self.to3d, self.expand3d, self.fusion_types):
            B, C, H, W = ap_f.shape
            if t == "attn":
                a = ap_f.flatten(2).transpose(1, 2); b = lat_f.flatten(2).transpose(1, 2)
                f2d = fuse(a, b).transpose(1, 2).reshape(B, C, H, W)
            else:
                f2d = fuse(ap_f, lat_f)
            fused2d.append(f2d)                                  # kept only for feature-viz cells
            fused3d.append(self._ortho_lift(ap_f, lat_f, c2d, c3d))
        return fused2d, fused3d

print("encoder feature dims:", FEAT_DIMS)

## 2. Data - paired (DRR inputs, per-bone multi-label GT)

The (X-ray, CT) pairs are aligned *by construction*: the DRRs were rendered from the same knee CT
volumes whose bones were later segmented into per-bone STL meshes and voxelized (see
`00_gt_per_bone.ipynb`). For each DRR pair we look up the matching **per-bone ground truth**.

**Ground-truth target = 4-channel per-bone occupancy** (femur / tibia / patella / fibula), one
binary `{0,1}` channel per bone, from `data/interim/gt_per_bone_256/{dataset}/{key}/{key}_{bone}.nii.gz`,
resampled to `TARGET_RES^3` (nearest-neighbour) and stacked to `(4, T, T, T)`. This replaces the old
single-channel `predrr > GT_THRESH` occupancy. Pretraining the front-end against this same target is
what makes the frozen features useful to the U-Net/V-Net decoders downstream.

**Splitting** is done at the *knee* level (`dataset, case, side`); this notebook PRODUCES the per-fold
split CSV `03_decoder_pipeline.ipynb` then loads, so both use identical fold boundaries. We restrict to
knees that actually have per-bone GT on disk and exclude geometric augmentations by default.

In [ ]:
def build_paired_index():
    """One row per (case, side, variant) with absolute AP/LAT paths + metadata."""
    rows = []
    nmeta = pd.read_csv(NORMAL_DRR_DIR / "drr_generation_metadata.csv")
    for (ds, case, side), _ in nmeta.groupby(["dataset", "case", "side"]):
        ap = NORMAL_DRR_DIR / ds / case / side / "ap.npy"
        lat = NORMAL_DRR_DIR / ds / case / side / "lat.npy"
        if ap.exists() and lat.exists():
            rows.append(dict(dataset=ds, case=case, side=side, variant="normal",
                             geometric=False, ap=str(ap), lat=str(lat)))
    ameta_path = AUG_DRR_DIR / "augmentation_variants_metadata.csv"
    if ameta_path.exists():
        ameta = pd.read_csv(ameta_path)
        for r in ameta.itertuples(index=False):
            ap = AUG_DRR_DIR / r.ap_npy; lat = AUG_DRR_DIR / r.lat_npy
            if ap.exists() and lat.exists():
                rows.append(dict(dataset=r.dataset, case=r.case, side=r.side, variant=r.variant,
                                 geometric=bool(r.geometric), ap=str(ap), lat=str(lat)))
    return pd.DataFrame(rows)

# ---- per-bone GT key: fractured folders carry a "Part" token, VSD healthy do not ----
def key_from(dataset, case, side):
    Side = "Right" if str(side).lower().startswith("r") else "Left"
    return ("%s_Part%s" % (case, Side)) if dataset == "fractured" else ("%s_%s" % (case, Side))

def gt_pb_dir(dataset, case, side):
    return GT_PB_DIR / dataset / key_from(dataset, case, side)

def has_per_bone_gt(dataset, case, side):
    d = gt_pb_dir(dataset, case, side)
    return all((d / ("%s_%s.nii.gz" % (d.name, b))).exists() for b in BONES)

# ---- ground truth: 4-channel per-bone occupancy at TARGET_RES (cached as one .npy per knee) ----
def load_gt_per_bone(dataset, case, side):
    GT_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    cache = GT_CACHE_DIR / ("%s_%s_%s.npy" % (dataset, case, side))
    if cache.exists():
        arr = np.load(cache)                                  # (4, T, T, T) float32
    else:
        d = gt_pb_dir(dataset, case, side); chans = []
        for b in BONES:
            vol = nib.load(str(d / ("%s_%s.nii.gz" % (d.name, b)))).get_fdata().astype(np.float32)
            occ = (vol > 0.5).astype(np.float32)              # STL masks are already binary
            t = F.interpolate(torch.from_numpy(occ)[None, None], size=(TARGET_RES,) * 3, mode="nearest")
            chans.append(t[0, 0].numpy())
        arr = np.stack(chans).astype(np.float32)              # (4, T, T, T)
        np.save(cache, arr)
    return torch.from_numpy(arr)                              # (4, T, T, T)

USE_TSDF = (STEP2_TARGET == "tsdf_blend")   # C-1: also supply the per-bone TSDF target

def load_tsdf_per_bone(dataset, case, side):
    """4-channel per-bone TSDF in [-1,1] at TARGET_RES (trilinear-resampled, cached per knee).
    The signed field is a continuous surface-distance signal, so it is linearly interpolated
    (unlike the nearest-neighbour occupancy). Built by the C-1 section of 00_gt_per_bone.ipynb."""
    GT_TSDF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    cache = GT_TSDF_CACHE_DIR / ("%s_%s_%s.npy" % (dataset, case, side))
    if cache.exists():
        arr = np.load(cache)
    else:
        d = gt_pb_dir(dataset, case, side); chans = []
        for b in BONES:
            fp = d / ("%s_%s_tsdf.nii.gz" % (d.name, b))
            assert fp.exists(), ("missing TSDF %s - run the C-1 TSDF section in 00_gt_per_bone.ipynb" % fp.name)
            vol = nib.load(str(fp)).get_fdata().astype(np.float32)          # already in [-1, 1]
            t = F.interpolate(torch.from_numpy(vol)[None, None], size=(TARGET_RES,) * 3,
                              mode="trilinear", align_corners=False)
            chans.append(t[0, 0].clamp(-1, 1).numpy())
        arr = np.stack(chans).astype(np.float32)                            # (4, T, T, T)
        np.save(cache, arr)
    return torch.from_numpy(arr)

class PairedDRRVolumeDataset(Dataset):
    """Returns AP/LAT DRRs (3x256x256) + per-bone GT occupancy (4,T,T,T) + metadata."""
    def __init__(self, df, transform=paired_tf):
        self.df = df.reset_index(drop=True); self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        ap = self.transform(load_drr(r.ap)); lat = self.transform(load_drr(r.lat))
        gt = load_gt_per_bone(r.dataset, r.case, r.side)      # (4, T, T, T)
        tsdf = load_tsdf_per_bone(r.dataset, r.case, r.side) if USE_TSDF else torch.zeros(1)
        return {"ap": ap, "lat": lat, "gt": gt, "tsdf": tsdf,
                "dataset": r.dataset, "case": r.case, "side": r.side, "variant": r.variant}

paired_index = build_paired_index()
if not INCLUDE_GEOMETRIC:
    paired_index = paired_index[~paired_index.geometric].reset_index(drop=True)
# restrict to knees that have per-bone GT built on disk (both cohorts)
paired_index = paired_index[paired_index.apply(
    lambda r: has_per_bone_gt(r.dataset, r.case, r.side), axis=1)].reset_index(drop=True)
print("paired rows (with per-bone GT):", len(paired_index),
      "| knees:", paired_index.groupby(["dataset", "case", "side"]).ngroups,
      "| cases:", paired_index.groupby("dataset")["case"].nunique().to_dict())

# ---- knee-level K-FOLD cross-validation split (dataset-stratified) ----
# This notebook PRODUCES the per-fold split CSV the decoder loads. 5-fold CV (as in FracReconNet,
# PMC9829664) over (case, side) puts every knee in a test fold exactly once. Round-robin slicing
# keeps fold sizes balanced; test = fold k, val = fold (k+1), train = the rest.
def kfold_split(df, n_folds=N_FOLDS, fold=FOLD, seed=SEED):
    rng = random.Random(seed); assign = {}
    for ds, g in df.groupby("dataset"):
        keys = sorted({(r.case, r.side) for r in g.itertuples()})
        rng.shuffle(keys)
        folds = [keys[i::n_folds] for i in range(n_folds)]      # round-robin -> balanced sizes
        test_keys = set(folds[fold % n_folds]); val_keys = set(folds[(fold + 1) % n_folds])
        for k in keys:
            assign[(ds,) + k] = "test" if k in test_keys else ("val" if k in val_keys else "train")
    return df.apply(lambda r: assign[(r.dataset, r.case, r.side)], axis=1)

if SMOKE_TEST:
    # tiny, fast subset just to verify the pipeline runs end-to-end (keep FOLD=0 in smoke)
    keep = paired_index[paired_index.variant == "normal"]
    sub = [g[g.case.isin(list(dict.fromkeys(g.case))[:SMOKE_CASES_PER_GROUP])]
           for _, g in keep.groupby("dataset")]
    paired_index = pd.concat(sub).reset_index(drop=True)

assert 0 <= FOLD < N_FOLDS, (
    "FOLD must be in [0, %d]; got %d. kfold_split uses fold %% N_FOLDS, which silently "
    "aliases an out-of-range FOLD onto another fold and double-counts its knees in the "
    "gate CSV." % (N_FOLDS - 1, FOLD))
paired_index["split"] = kfold_split(paired_index)
split_csv = MODELS_DIR / "decoders" / ("decoder_split_fold%d.csv" % FOLD)
split_csv.parent.mkdir(parents=True, exist_ok=True)
paired_index[["dataset", "case", "side", "variant", "split"]].to_csv(split_csv, index=False)
print("wrote fold %d split -> %s" % (FOLD, split_csv.name))
print(paired_index.groupby(["split", "dataset"]).size())

In [ ]:
train_df = paired_index[paired_index.split == "train"]
# Evaluate on the CLEAN DRR only: photometric augmented variants are a TRAIN-time augmentation.
# Keeping them in val/test would duplicate each held-out knee as several near-identical rows,
# making the per-knee metric (and the U-Net vs V-Net pairing downstream) ambiguous.
val_df   = paired_index[(paired_index.split == "val")  & (paired_index.variant == "normal")]
test_df  = paired_index[(paired_index.split == "test") & (paired_index.variant == "normal")]

def make_loader(df, shuffle):
    if len(df) == 0:
        return None
    return DataLoader(PairedDRRVolumeDataset(df), batch_size=BATCH_SIZE,
                      shuffle=shuffle, num_workers=NUM_WORKERS, drop_last=False)

train_loader = make_loader(train_df, True)
val_loader   = make_loader(val_df, False)
test_loader  = make_loader(test_df, False)
print("samples -> train:", len(train_df), "| val:", len(val_df), "| test:", len(test_df),
      "(val/test = normal variant only)")

## 3. Neutral pretraining head

The head reuses the **exact decoder wiring** both contenders share — upsample step by step
(`8 -> 16 -> 32 -> 64`) with encoder features concatenated as **skip connections**, then a
**super-resolution head** grows the `(LIFT_DEPTH, 64, 64)` grid up to `TARGET_RES^3`. The **only**
difference from the real decoders is the building block:

- **`neutral`** -> `SingleConv`: one `Conv3d -> BatchNorm -> ReLU`.
- (`unet` -> `DoubleConv` = two of these; `vnet` -> `VNetResBlock` = two + residual — defined here
  too, but **not used** in this notebook.)

Training with `SingleConv` gives the frozen encoder's fusion + 2D→3D lift a gradient signal toward
the occupancy task without tuning them to the quirks of either decoder. After training we save the
front-end and **discard this head**.

In [ ]:
def conv_block(block_type, in_ch, out_ch):
    if block_type == "unet":
        return DoubleConv(in_ch, out_ch)
    if block_type == "vnet":
        return VNetResBlock(in_ch, out_ch)
    if block_type == "neutral":
        return SingleConv(in_ch, out_ch)
    raise ValueError("unknown block_type: %s" % block_type)

class SingleConv(nn.Module):
    """Neutral block: a single Conv3d -> BN -> ReLU. The common ancestor of U-Net's DoubleConv
    (two of these) and V-Net's residual block (two + a skip), so pretraining the front-end with it
    favours neither decoder. Used ONLY to give the fusion + 2D->3D lift a gradient signal; the head
    itself is discarded after pretraining."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1), nn.BatchNorm3d(out_ch), nn.ReLU(inplace=True))
    def forward(self, x):
        return self.net(x)

class DoubleConv(nn.Module):
    """U-Net block: (Conv3d -> BN -> ReLU) x2. Plain, no residual."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1), nn.BatchNorm3d(out_ch), nn.ReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1), nn.BatchNorm3d(out_ch), nn.ReLU(inplace=True))
    def forward(self, x):
        return self.net(x)

class VNetResBlock(nn.Module):
    """V-Net block: (Conv3d -> BN -> PReLU) x2 + residual add (input projected if channels differ)."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.proj = nn.Conv3d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
        self.c1 = nn.Conv3d(in_ch, out_ch, 3, padding=1); self.n1 = nn.BatchNorm3d(out_ch); self.a1 = nn.PReLU(out_ch)
        self.c2 = nn.Conv3d(out_ch, out_ch, 3, padding=1); self.n2 = nn.BatchNorm3d(out_ch); self.a2 = nn.PReLU(out_ch)
    def forward(self, x):
        y = self.a1(self.n1(self.c1(x)))
        y = self.n2(self.c2(y))
        return self.a2(y + self.proj(x))

class SuperResHead(nn.Module):
    """Grow the (S, S, S) cube feature grid up to (T,T,T) via staged trilinear upsample + refine
    blocks with tapering channels (heavy work stays at low resolution -> low memory).
    Output: (B, N_CLASSES, T, T, T) - one logit map per bone."""
    def __init__(self, block_type, in_ch, target, use_grad_ckpt=False):
        super().__init__()
        self.target = tuple(int(t) for t in target); self.use_grad_ckpt = use_grad_ckpt
        self.b1 = conv_block(block_type, in_ch, 32)
        self.b2 = conv_block(block_type, 32, 16)
        self.b3 = conv_block(block_type, 16, 8)
        self.out = nn.Conv3d(8, N_CLASSES, 1)   # multi-label: one channel per bone (was 1)
    def _run(self, blk, x):
        if self.use_grad_ckpt and x.requires_grad:
            return cp.checkpoint(blk, x, use_reentrant=False)
        return blk(x)
    def forward(self, x):
        d0, h0, w0 = x.shape[-3:]; dt, ht, wt = self.target
        s1 = (round(d0 + (dt - d0) / 3), round(h0 + (ht - h0) / 3), round(w0 + (wt - w0) / 3))
        s2 = (round(d0 + 2 * (dt - d0) / 3), round(h0 + 2 * (ht - h0) / 3), round(w0 + 2 * (wt - w0) / 3))
        x = F.interpolate(x, size=s1, mode="trilinear", align_corners=False); x = self._run(self.b1, x)
        x = F.interpolate(x, size=s2, mode="trilinear", align_corners=False); x = self._run(self.b2, x)
        x = F.interpolate(x, size=self.target, mode="trilinear", align_corners=False); x = self._run(self.b3, x)
        return self.out(x)

class Decoder3D(nn.Module):
    """Multi-scale skip-connected decoder. Same wiring for every block type; only the block differs
    (unet=DoubleConv, vnet=VNetResBlock, neutral=SingleConv). Inputs are CUBE features from the
    orthogonal lift (l3=8^3 ... l0=64^3), so the upsamplers are symmetric stride-2 on all three
    axes (8->16->32->64), matching each skip; SuperResHead then grows 64^3 -> T^3.
    Output: (B, N_CLASSES, T, T, T)."""
    def __init__(self, block_type, enc_channels=OUT_CHANNELS, target=(64, 64, 64),
                 use_grad_ckpt=False, deep_supervision=False):
        super().__init__()
        c0, c1, c2, c3 = enc_channels
        self.deep_supervision = deep_supervision; self.target = tuple(int(t) for t in target)
        self.up3 = nn.ConvTranspose3d(c3, c2, kernel_size=2, stride=2)   # 8^3 -> 16^3 (symmetric)
        self.dec3 = conv_block(block_type, c2 + c2, c2)
        self.up2 = nn.ConvTranspose3d(c2, c1, kernel_size=2, stride=2)   # 16^3 -> 32^3
        self.dec2 = conv_block(block_type, c1 + c1, c1)
        self.up1 = nn.ConvTranspose3d(c1, c0, kernel_size=2, stride=2)   # 32^3 -> 64^3
        self.dec1 = conv_block(block_type, c0 + c0, c0)
        self.sr = SuperResHead(block_type, c0, self.target, use_grad_ckpt)
        if deep_supervision:
            self.aux3 = nn.Conv3d(c2, N_CLASSES, 1); self.aux2 = nn.Conv3d(c1, N_CLASSES, 1); self.aux1 = nn.Conv3d(c0, N_CLASSES, 1)
    def forward(self, feats):
        l0, l1, l2, l3 = feats
        x = self.up3(l3); x = torch.cat([x, l2], 1); x = self.dec3(x); a3 = x
        x = self.up2(x);  x = torch.cat([x, l1], 1); x = self.dec2(x); a2 = x
        x = self.up1(x);  x = torch.cat([x, l0], 1); x = self.dec1(x); a1 = x
        out = self.sr(x)
        if self.deep_supervision and self.training:
            up = lambda h: F.interpolate(h, size=self.target, mode="trilinear", align_corners=False)
            return out, [up(self.aux3(a3)), up(self.aux2(a2)), up(self.aux1(a1))]
        return out, None

class ReconModel(nn.Module):
    """Full model = shared bi-planar encoder/fusion + a decoder (here the neutral pretraining head)."""
    def __init__(self, fusion, decoder):
        super().__init__(); self.fusion = fusion; self.decoder = decoder
    def forward(self, ap, lat):
        _, f3d = self.fusion(ap, lat)
        return self.decoder(f3d)

In [ ]:
def build_model():
    fusion = BiPlanarFeatureFusion(depth=LIFT_DEPTH, pretrained=PRETRAINED, freeze_encoder=FREEZE_ENCODER)
    if ENCODER_CKPT.exists():
        fusion.load_simclr_encoder(ENCODER_CKPT)   # loads whatever encoder state_dict is at ENCODER_CKPT (FCMAE preferred)
        print("[encoder] init from", ENCODER_CKPT.name)
    else:
        print("[warn] no encoder checkpoint (%s); encoder uses ImageNet/random init." % ENCODER_CKPT.name)
    decoder = Decoder3D(MODEL, target=(TARGET_RES,) * 3,
                        use_grad_ckpt=USE_GRAD_CKPT, deep_supervision=DEEP_SUPERVISION)
    return ReconModel(fusion, decoder)

# shape sanity check (eval mode, no grad -> cheap)
_m = build_model().to(DEVICE).eval()
with torch.no_grad():
    _ap = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
    _out, _ = _m(_ap, _ap)
print("model:", MODEL, "| output volume:", tuple(_out.shape),
      "| expected:", (1, N_CLASSES, TARGET_RES, TARGET_RES, TARGET_RES))
del _m, _ap, _out

## Diagnostics — smoking-gun checks (assert + visualize)

Two guards that would have caught the extrusion bug (3D features constant along depth → Dice stuck
at ~0.45) and that fail loudly on a regression:

- **Check 1 — GT↔DRR axis alignment.** Back-projects both DRRs through the lift's own geometry and
  asserts their intersection actually contains the bone (recall > 0.30). Also shows the two DRRs
  next to the GT projections so you can eyeball that AP↔axis1 and LAT↔axis0.
- **Check 2 — per-axis feature variance.** Asserts the 3D features vary along **all three** spatial
  axes (std > 1e-4). The old extruding lift produced std≈0 along depth — this is the direct detector.

In [ ]:
# ===== Check 1 - GT<->DRR axis alignment (visual + hull-recall guard) =====
# Back-project both DRRs exactly as _ortho_lift does (O=1, no conv) and intersect them. With the
# configured orientation the bone must fall INSIDE the hull -> high recall. A gross axis/flip error
# separates the two shadows and recall collapses. GT here is the per-bone UNION (max over the 4 bone
# channels). (Subtle flips were settled by a one-sample overfit test; this guards gross mis-orientation.)
@torch.no_grad()
def _hull_recall(ap2d, lat2d, occ, res=96):
    a = torch.from_numpy(ap2d)[None, None].float(); l = torch.from_numpy(lat2d)[None, None].float()
    a = F.interpolate(a, (res, res), mode="bilinear", align_corners=False)
    l = F.interpolate(l, (res, res), mode="bilinear", align_corners=False)
    a = (a - a.min()) / (a.max() - a.min() + 1e-8); l = (l - l.min()) / (l.max() - l.min() + 1e-8)
    if LIFT_FLIP_SI:      a = a.flip(2); l = l.flip(2)
    if LIFT_FLIP_AP_COL:  a = a.flip(3)
    if LIFT_FLIP_LAT_COL: l = l.flip(3)
    S = res
    ac = a.permute(0, 1, 3, 2).unsqueeze(3).expand(1, 1, S, S, S)   # AP  -> broadcast axis1 (A-P)
    lc = l.permute(0, 1, 3, 2).unsqueeze(2).expand(1, 1, S, S, S)   # LAT -> broadcast axis0 (L-R)
    hull = (ac * lc)[0, 0].numpy()
    occ_r = F.interpolate(torch.from_numpy(occ.astype(np.float32))[None, None], (S, S, S),
                          mode="nearest")[0, 0].numpy() > 0.5
    k = max(int(3 * occ_r.sum()), 1)                                # keep top ~3x bone-count voxels
    thr = np.partition(hull.ravel(), -k)[-k]
    return ((hull >= thr) & occ_r).sum() / max(occ_r.sum(), 1)

_s = paired_index.iloc[0]
_occ = load_gt_per_bone(_s.dataset, _s.case, _s.side).numpy().max(0) > 0.5   # per-bone UNION
_ap_img = np.load(_s.ap).astype(np.float32); _lat_img = np.load(_s.lat).astype(np.float32)
_rec = _hull_recall(_ap_img, _lat_img, _occ)
print("Check 1 - back-projected hull recall of GT = %.3f  (orientation SI=%s AP=%s LAT=%s; case %s %s)"
      % (_rec, LIFT_FLIP_SI, LIFT_FLIP_AP_COL, LIFT_FLIP_LAT_COL, _s.case, _s.side))
fig, ax = plt.subplots(1, 5, figsize=(16, 3.2))
ax[0].imshow(_ap_img, cmap="gray"); ax[0].set_title("AP DRR")
ax[1].imshow(_lat_img, cmap="gray"); ax[1].set_title("LAT DRR")
for j, k in enumerate((0, 1, 2)):
    ax[2 + j].imshow(_occ.max(axis=k), cmap="gray"); ax[2 + j].set_title("GT MIP axis%d" % k)
for a in ax:
    a.axis("off")
plt.suptitle("Check 1 - inputs vs GT projections (lift expects AP->axis1, LAT->axis0)")
plt.tight_layout(); plt.show()
assert _rec > 0.30, "Check 1 FAIL: configured back-projection misses the bone (recall<0.30) - orientation likely wrong"
print("Check 1 PASS: the configured orthogonal back-projection contains the bone.")

In [ ]:
# ===== Check 2 - per-axis feature variance (the extrusion detector) =====
# The reference lift broadcast one fused 2D map along depth -> std==0 along that axis (the bug that
# capped Dice ~0.45). The orthogonal lift must vary along ALL THREE spatial axes. FAIL loudly if any
# axis collapses, so a regression to an extruding lift cannot pass a run silently.
_diag = build_model().to(DEVICE).eval()
_rows = []
with torch.no_grad():
    for ds in ["healthy", "fractured"]:
        sub = paired_index[paired_index.dataset == ds]
        if len(sub) == 0:
            continue
        r = sub.iloc[0]
        ap = paired_tf(load_drr(r.ap)).unsqueeze(0).to(DEVICE)
        lat = paired_tf(load_drr(r.lat)).unsqueeze(0).to(DEVICE)
        _, f3d = _diag.fusion(ap, lat)
        for i, f in enumerate(f3d):
            s0, s1, s2 = f.std(2).mean().item(), f.std(3).mean().item(), f.std(4).mean().item()
            _rows.append((ds, i, s0, s1, s2, min(s0, s1, s2) > 1e-4))
print("Check 2 - per-axis feature std (must be > 1e-4 on every axis):")
for ds, i, s0, s1, s2, ok in _rows:
    print("  %-9s level%d: std[axis0]=%.4f std[axis1]=%.4f std[axis2]=%.4f  %s"
          % (ds, i, s0, s1, s2, "OK" if ok else "FAIL - axis collapsed (extrusion!)"))
assert _rows and all(r[5] for r in _rows), "Check 2 FAIL: a 3D feature is constant along an axis (extrusion bug)"
print("Check 2 PASS: 3D features vary on all three axes (no extrusion).\n")
del _diag

In [ ]:
# --- NaN diagnostic (optional): finds the first non-finite tensor. Set False to skip. ---
# If training ever prints NaN, run this: it shows whether the inputs, the fused 3D features, or the
# logits become non-finite, and whether float16 autocast (vs fp32) is what introduces it.
RUN_NAN_DIAGNOSTIC = True
if RUN_NAN_DIAGNOSTIC and train_loader is not None:
    _dm = build_model().to(DEVICE).eval()
    _b = next(iter(train_loader))
    _ap, _lat, _gt = _b["ap"].to(DEVICE), _b["lat"].to(DEVICE), _b["gt"].to(DEVICE)
    def _chk(name, t):
        print("  %-12s finite=%s min=%.3g max=%.3g"
              % (name, bool(torch.isfinite(t).all()), float(t.min()), float(t.max())))
    print("inputs:"); _chk("ap", _ap); _chk("lat", _lat); _chk("gt", _gt)
    for use in ([False, True] if DEVICE.type == "cuda" else [False]):
        ad = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        with torch.no_grad():
            if use:
                with torch.amp.autocast("cuda", dtype=ad):
                    _f2, _f3 = _dm.fusion(_ap, _lat); _o, _ = _dm.decoder(_f3)
            else:
                _f2, _f3 = _dm.fusion(_ap, _lat); _o, _ = _dm.decoder(_f3)
        print("autocast", use, "| dtype", (str(ad) if use else "fp32"))
        for i, f in enumerate(_f3):
            _chk("feat3d[%d]" % i, f)
        _chk("logits", _o)
    del _dm, _b, _ap, _lat, _gt

## 4. Loss and metrics

- **Loss = 0.5 * BCE + 0.5 * soft-Dice.** Bone is a small fraction of the volume (~3%), so pure BCE
  is dominated by easy background voxels. Adding Dice directly optimises overlap and handles the
  class imbalance. (Lai's reference used Dice only; adding BCE stabilises early training.)
- **Metrics:** **Dice** and **IoU** (overlap), and **HD95** (95th-percentile surface distance, in
  voxels) for boundary accuracy. We compute them on the *binarised* prediction. We report them
  **overall and split by healthy vs fractured**, which is what the research goal needs.

We deliberately implement these by hand (instead of pulling in MONAI) so the formulas are visible
and the notebook runs with the libraries already installed.

In [ ]:
class DiceBCEMC(nn.Module):
    """Multi-channel BCE + soft-Dice, averaged over the N_CLASSES bone channels. Reductions run in
    float32 so a 256^3 sum under AMP float16 cannot overflow (>65504 -> inf -> NaN)."""
    def __init__(self, w_bce=0.5, w_dice=0.5, smooth=1.0):
        super().__init__(); self.w_bce = w_bce; self.w_dice = w_dice; self.smooth = smooth
    def _dice(self, logits, target):
        p = torch.sigmoid(logits.float()); t = target.float()
        p = p.reshape(p.size(0), p.size(1), -1); t = t.reshape(t.size(0), t.size(1), -1)
        inter = (p * t).sum(-1)
        d = (2 * inter + self.smooth) / (p.sum(-1) + t.sum(-1) + self.smooth)
        return 1 - d.mean()
    def forward(self, logits, target):
        logits = logits.float()   # compute the loss in fp32 even when the forward pass ran in fp16
        return self.w_bce * F.binary_cross_entropy_with_logits(logits, target.float()) + self.w_dice * self._dice(logits, target)

DICE_BCE = DiceBCEMC()

W_TSDF, W_BIN = 0.5, 0.5   # C-1 blend weights (user-selected 50/50)

def tsdf_l1_band(logits, tsdf_gt):
    """C-1: L1 between the predicted signed field and the TSDF target, restricted to the
    truncation band (|tsdf|<1), per bone channel then averaged. The occupancy head is
    reparametrized into signed space s = 1 - 2*sigmoid(z) (inside->-1, outside->+1,
    surface->0) so no extra head is needed; this concentrates gradient on near-boundary
    voxels - the surface the HD95/ASSD metrics turn on."""
    s_pred = 1.0 - 2.0 * torch.sigmoid(logits.float())      # (B,C,...) in (-1,1)
    tgt = tsdf_gt.float()
    band = (tgt.abs() < 1.0).float()                        # near-surface voxels only
    num = (band * (s_pred - tgt).abs()).flatten(2).sum(-1)
    den = band.flatten(2).sum(-1).clamp_min(1.0)
    return (num / den).mean()                               # mean over batch & channels

def total_loss(output, target, tsdf=None, aux_weight=0.3):
    out, aux = output
    binary = DICE_BCE(out, target)
    loss = (W_TSDF * tsdf_l1_band(out, tsdf) + W_BIN * binary) \
           if (USE_TSDF and tsdf is not None) else binary
    if aux:
        for a in aux:
            loss = loss + aux_weight * DICE_BCE(a, target)
    return loss

@torch.no_grad()
def eval_loss_terms(model, loader):
    """M-11: mean step-2 loss split by term (binary BCE/Dice vs TSDF-L1) over a loader."""
    model.eval(); tb = tt = n = 0.0
    for batch in loader:
        out, _ = model(batch['ap'].to(DEVICE), batch['lat'].to(DEVICE))
        bsz = batch['ap'].size(0); n += bsz
        tb += DICE_BCE(out, batch['gt'].to(DEVICE)).item() * bsz
        if USE_TSDF:
            tt += tsdf_l1_band(out, batch['tsdf'].to(DEVICE)).item() * bsz
    n = max(n, 1.0)
    return tb / n, (tt / n if USE_TSDF else float('nan'))

@torch.no_grad()
def dice_iou(logits, target, thr=0.5):
    """Per-knee Dice/IoU, each averaged over the N_CLASSES bone channels -> (B,) arrays."""
    p = (torch.sigmoid(logits.float()) > thr).float()
    t = (target > 0.5).float()
    p = p.reshape(p.size(0), p.size(1), -1); t = t.reshape(t.size(0), t.size(1), -1)
    inter = (p * t).sum(-1); psum = p.sum(-1); tsum = t.sum(-1)
    dice = ((2 * inter + 1e-6) / (psum + tsum + 1e-6)).mean(1)
    iou = ((inter + 1e-6) / (psum + tsum - inter + 1e-6)).mean(1)
    return dice.cpu().numpy(), iou.cpu().numpy()

In [ ]:
def run_epoch(model, loader, optimizer, scaler, train):
    model.train(train)
    use_amp = USE_AMP and DEVICE.type == "cuda"
    # bfloat16 has float32's dynamic range -> no overflow at 65504 (the float16 NaN cause).
    amp_dtype = torch.bfloat16 if (use_amp and torch.cuda.is_bf16_supported()) else torch.float16
    use_scaler = use_amp and amp_dtype == torch.float16   # only float16 needs GradScaler
    tot, n, steps, skipped = 0.0, 0, 0, 0
    for batch in loader:
        ap = batch["ap"].to(DEVICE); lat = batch["lat"].to(DEVICE); gt = batch["gt"].to(DEVICE)
        tsdf = batch["tsdf"].to(DEVICE) if USE_TSDF else None
        with torch.set_grad_enabled(train):
            if use_amp:
                with torch.amp.autocast("cuda", dtype=amp_dtype):
                    loss = total_loss(model(ap, lat), gt, tsdf)
            else:
                loss = total_loss(model(ap, lat), gt, tsdf)
        if not torch.isfinite(loss):                       # never backprop a NaN/inf loss
            skipped += 1; optimizer.zero_grad(set_to_none=True); continue
        if train:
            optimizer.zero_grad(set_to_none=True)
            if use_scaler:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                prev = scaler.get_scale(); scaler.step(optimizer); scaler.update()
                if scaler.get_scale() >= prev: steps += 1   # scaler did not skip the step
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step(); steps += 1
        tot += loss.item() * ap.size(0); n += ap.size(0)
    if skipped: print("  [warn] skipped %d non-finite batch(es)" % skipped)
    return tot / max(n, 1), steps

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); rows = []
    for batch in loader:
        out, _ = model(batch["ap"].to(DEVICE), batch["lat"].to(DEVICE))
        d, i = dice_iou(out, batch["gt"].to(DEVICE))
        for b in range(len(d)):
            rows.append(dict(dataset=batch["dataset"][b], case=batch["case"][b],
                             side=batch["side"][b], dice=float(d[b]), iou=float(i[b])))
    df = pd.DataFrame(rows)
    overall = df[["dice", "iou"]].mean().to_dict() if len(df) else {"dice": float("nan"), "iou": float("nan")}
    by = df.groupby("dataset")[["dice", "iou"]].mean() if len(df) else None
    return df, overall, by

def save_ckpt(path, model, optimizer, scheduler, epoch, val_metrics):
    torch.save({"epoch": epoch, "model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict() if scheduler else None, "val_metrics": val_metrics,
                "config": {"MODEL": MODEL, "TARGET_RES": TARGET_RES, "LIFT_DEPTH": LIFT_DEPTH,
                           "N_CLASSES": N_CLASSES, "BONES": BONES}}, path)

## 5. Training with checkpointing

Each epoch we train, validate, score Dice/IoU on the validation set, step the cosine LR schedule,
and **save checkpoints**:
- `MODEL_last.pth` — always the most recent (for resuming),
- `MODEL_epochNNN.pth` — every `CKPT_EVERY` epochs (so you can **revisit any epoch** later),
- `MODEL_best.pth` — whenever validation Dice improves,
- `MODEL_history.csv` — per-epoch losses/metrics for the learning-curve plot.

Each checkpoint stores the epoch, model + optimizer + scheduler state, the validation metrics, and
the run config (model type, resolution, lift depth, GT threshold) — everything needed to resume or
to load the model later in the UI. On GPU we use **AMP** (mixed precision) and optional **gradient
checkpointing** to fit `256^3` in memory.

In [ ]:
model = build_model().to(DEVICE)
# encoder is frozen (FREEZE_ENCODER); train only the fusion + 2D->3D lift + neutral head
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(EPOCHS, 1))
scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and DEVICE.type == "cuda"))

n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print("trainable params: %.2fM (fusion+lift+neutral head) | frozen: %.2fM (encoder)"
      % (n_train / 1e6, n_frozen / 1e6))

start_epoch, best_dice, history = 0, -1.0, []
if RESUME_FROM:
    ck = torch.load(RESUME_FROM, map_location=DEVICE)
    model.load_state_dict(ck["model"]); optimizer.load_state_dict(ck["optimizer"])
    if ck.get("scheduler"):
        scheduler.load_state_dict(ck["scheduler"])
    start_epoch = ck["epoch"] + 1
    print("resumed from %s at epoch %d" % (RESUME_FROM, start_epoch))

t0 = time.time()
for epoch in range(start_epoch, EPOCHS):
    tr, steps = run_epoch(model, train_loader, optimizer, scaler, train=True)
    if val_loader:
        va, _ = run_epoch(model, val_loader, optimizer, scaler, train=False)
        _, overall, _ = evaluate(model, val_loader)
    else:
        va, overall = float("nan"), {"dice": float("nan"), "iou": float("nan")}
    if USE_TSDF and val_loader:                       # M-11: val loss split by target type
        m11_bin, m11_tsdf = eval_loss_terms(model, val_loader)
    else:
        m11_bin, m11_tsdf = float('nan'), float('nan')
    if steps > 0:                 # optimizer.step() ran this epoch -> correct order to step scheduler
        scheduler.step()
    history.append(dict(epoch=epoch, train_loss=tr, val_loss=va, val_dice=overall["dice"],
                        val_iou=overall["iou"], val_binary=m11_bin, val_tsdf=m11_tsdf,
                        lr=optimizer.param_groups[0]["lr"], secs=round(time.time() - t0, 1)))
    pd.DataFrame(history).to_csv(CKPT_DIR / ("%s_history.csv" % MODEL), index=False)
    save_ckpt(CKPT_DIR / ("%s_last.pth" % MODEL), model, optimizer, scheduler, epoch, overall)
    if epoch % CKPT_EVERY == 0:
        save_ckpt(CKPT_DIR / ("%s_epoch%03d.pth" % (MODEL, epoch)), model, optimizer, scheduler, epoch, overall)
    if overall["dice"] > best_dice:
        best_dice = overall["dice"]
        save_ckpt(CKPT_DIR / ("%s_best.pth" % MODEL), model, optimizer, scheduler, epoch, overall)
    print("epoch %03d | train %.4f | val %.4f | val_dice %.4f | best %.4f"
          % (epoch, tr, va, overall["dice"], best_dice))
    if USE_TSDF:
        print("    [M-11] val binary=%.4f | tsdf=%.4f" % (m11_bin, m11_tsdf))
print("done. checkpoints in", CKPT_DIR)

## 6. Save the front-end + sanity check

We reload the **best** epoch (lowest val loss / best val Dice), then save the front-end (frozen
encoder + trained fusion + 2D→3D lift) to `models/front_end.pth` — the artifact the decoder pipeline
loads and freezes. The neutral head is discarded. The Dice/IoU below are just a sanity check that
the front-end learned something useful (a rough smoke value is fine here); the real numbers come
from the decoder runs.

In [ ]:
best_path = CKPT_DIR / ("%s_best.pth" % MODEL)
if best_path.exists():
    model.load_state_dict(torch.load(best_path, map_location=DEVICE)["model"])
    print("loaded", best_path.name)

# ---- save the trained front-end (frozen encoder + trained fusion + 2D->3D lift) ----
# This is the artifact 03_decoder_pipeline.ipynb loads and freezes wholesale. The neutral head is
# NOT saved -- it was only a scaffold to give the fusion/lift a gradient signal.
torch.save({"front_end": model.fusion.state_dict(),
            "config": {"LIFT_DEPTH": LIFT_DEPTH, "BACKBONE": BACKBONE,
                       "TARGET_RES": TARGET_RES, "N_CLASSES": N_CLASSES, "BONES": BONES}},
           FRONTEND_CKPT)
print("saved front-end ->", FRONTEND_CKPT)

# ---- sanity-check the neutral head (overall + healthy vs fractured) ----
eval_loader = test_loader or val_loader
if eval_loader:
    df, overall, by = evaluate(model, eval_loader)
    print("OVERALL:", {k: round(v, 4) for k, v in overall.items()})
    if by is not None:
        print("\nBY GROUP (healthy vs fractured):\n", by.round(4))
else:
    print("no eval data in this (smoke) split.")

In [ ]:
# ============================ Gate G4 arm metrics (C-6) ============================
# Neutral-head Dice + surface distance (HD95/ASSD in mm) on this fold's HELD-OUT knees, split by
# cohort, appended to a shared arm-metrics CSV. 04_decoder_comparison.ipynb reads this to arbitrate
# the front-end changes (C-1 TSDF / C-2 aux heads / C-3 unfreeze) one axis at a time; the IRREGULAR
# (fractured) cohort is decisive. Surface-metric code is the same implementation as 03.
from scipy.ndimage import binary_erosion, distance_transform_edt

def _surface_dists(pred_bin, gt_bin, spacing):
    sp = pred_bin & ~binary_erosion(pred_bin); sg = gt_bin & ~binary_erosion(gt_bin)
    if sp.sum() == 0 or sg.sum() == 0:
        return None
    dg = distance_transform_edt(~sg) * spacing; dp = distance_transform_edt(~sp) * spacing
    return dg[sp], dp[sg]

def _hd95(pb, gb, sp):
    d = _surface_dists(pb, gb, sp)
    return float("nan") if d is None else float(np.percentile(np.concatenate(d), 95))

def _assd(pb, gb, sp):
    d = _surface_dists(pb, gb, sp)
    return float("nan") if d is None else float(np.concatenate(d).mean())

def _voxel_mm():
    try:
        r = paired_index.iloc[0]; dd = gt_pb_dir(r.dataset, r.case, r.side)
        img = nib.load(str(dd / ("%s_%s.nii.gz" % (dd.name, BONES[0]))))
        zooms = np.asarray(img.header.get_zooms()[:3], float); dims = np.asarray(img.shape[:3], float)
        return float(np.mean(zooms * dims / TARGET_RES))
    except Exception:
        return 0.78125 * 256.0 / TARGET_RES
VOXEL_MM = _voxel_mm()

# arm identity = the one axis being varied (C-6 changes exactly one at a time)
ARM = "%s|%s|%s" % ("tsdf" if USE_TSDF else "binary",
                    "unfrozen" if not FREEZE_ENCODER else "frozen",
                    "aux" if globals().get("USE_AUX", False) else "noaux")
# Staleness guard: USE_TSDF is cached from STEP2_TARGET at the moment the data-loader cell
# last ran. Editing STEP2_TARGET in CONFIG and re-running only cells below it (instead of
# Restart Kernel & Run All) leaves USE_TSDF stale, silently mislabeling (or mis-training)
# this arm. Fail loudly instead of writing an untrustworthy row to the gate CSV.
assert USE_TSDF == (STEP2_TARGET == "tsdf_blend"), (
    "USE_TSDF is stale relative to STEP2_TARGET (USE_TSDF=%s, STEP2_TARGET=%s). "
    "Restart the kernel and Run All before trusting this arm's gate metrics."
    % (USE_TSDF, STEP2_TARGET))

@torch.no_grad()
def gate_eval(model, loader):
    model.eval(); rows = []
    for batch in loader:
        out, _ = model(batch["ap"].to(DEVICE), batch["lat"].to(DEVICE))
        prob = torch.sigmoid(out.float()).cpu().numpy(); gtn = batch["gt"].numpy()
        dsc, _iou = dice_iou(out, batch["gt"].to(DEVICE))          # (B,) bone-averaged Dice
        for b in range(prob.shape[0]):
            hs, as_ = [], []
            for k in range(len(BONES)):
                pb = prob[b, k] > 0.5; gb = gtn[b, k] > 0.5
                hs.append(_hd95(pb, gb, VOXEL_MM)); as_.append(_assd(pb, gb, VOXEL_MM))
            rows.append(dict(arm=ARM, step2_target=STEP2_TARGET, frozen=bool(FREEZE_ENCODER), fold=FOLD,
                             dataset=batch["dataset"][b], case=batch["case"][b], side=batch["side"][b],
                             dice=float(dsc[b]), hd95_mm=float(np.nanmean(hs)), assd_mm=float(np.nanmean(as_))))
    return pd.DataFrame(rows)

gate_loader = test_loader or val_loader
if gate_loader:
    gdf = gate_eval(model, gate_loader)
    GATE_CSV = MODELS_DIR / "decoders" / "gate_g4_arm_metrics.csv"
    GATE_CSV.parent.mkdir(parents=True, exist_ok=True)
    if GATE_CSV.exists():                          # replace prior rows for THIS (arm, fold) on re-run
        prev = pd.read_csv(GATE_CSV)
        prev = prev[~((prev.arm == ARM) & (prev.fold == FOLD))]
        gdf = pd.concat([prev, gdf], ignore_index=True)
    gdf.to_csv(GATE_CSV, index=False)
    n_arm = int((gdf.arm == ARM).sum())
    print("[Gate G4] arm=%s fold=%d | recorded %d held-out knees -> %s" % (ARM, FOLD, n_arm, GATE_CSV.name))
    print(gdf[gdf.arm == ARM].groupby("dataset")[["dice", "hd95_mm", "assd_mm"]].mean().round(3).to_string())
else:
    print("[Gate G4] no held-out knees in this (smoke) split; nothing to record.")

In [ ]:
h = pd.read_csv(CKPT_DIR / ("%s_history.csv" % MODEL))
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(h.epoch, h.train_loss, label="train"); ax[0].plot(h.epoch, h.val_loss, label="val")
ax[0].set_title("%s loss" % MODEL); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(h.epoch, h.val_dice, label="val Dice"); ax[1].plot(h.epoch, h.val_iou, label="val IoU")
ax[1].set_title("%s val metrics" % MODEL); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.tight_layout(); plt.show()

## 7. Quality-assurance views

**(A) Per-bone GT preview.** The four stacked bone channels (femur/tibia/patella/fibula) for one
knee, with each channel's occupancy fraction - a quick check that all four bones are present and
sensibly sized (patella/fibula small; femur/tibia large).

**(B) Reconstruction preview.** Mid-slices of the neutral head's predicted bone *union* next to the
GT union, as a quick visual sanity check (rough after a 2-epoch smoke run - expected).

In [ ]:
# (A) per-bone GT preview: the 4 stacked bone channels for one knee (mid-coronal)
sample = paired_index.iloc[0]
gt4 = load_gt_per_bone(sample.dataset, sample.case, sample.side).numpy()   # (4, T, T, T)
mid = gt4.shape[2] // 2
fig, ax = plt.subplots(1, 4, figsize=(14, 4))
for j, bone in enumerate(BONES):
    ax[j].imshow(gt4[j, :, mid, :], cmap="gray", origin="lower")
    ax[j].set_title("%s  (%.2f%%)" % (bone, gt4[j].mean() * 100)); ax[j].axis("off")
plt.suptitle("%s %s %s - per-bone GT (mid-coronal)" % (sample.dataset, sample.case, sample.side))
plt.tight_layout(); plt.show()

# (B) reconstruction preview vs GT (bone union over the 4 channels)
if eval_loader:
    batch = next(iter(eval_loader))
    with torch.no_grad():
        out, _ = model(batch["ap"].to(DEVICE), batch["lat"].to(DEVICE))
    pred = (torch.sigmoid(out[0].float()).amax(0).cpu().numpy() > 0.5).astype(float)
    gtv = batch["gt"][0].numpy().max(0)
    m = pred.shape[0] // 2
    fig, ax = plt.subplots(1, 3, figsize=(11, 4))
    ax[0].imshow(batch["ap"][0, 0], cmap="gray"); ax[0].set_title("input AP")
    ax[1].imshow(gtv[m], cmap="gray"); ax[1].set_title("GT union mid-slice")
    ax[2].imshow(pred[m], cmap="gray"); ax[2].set_title("prediction union mid-slice")
    for a in ax:
        a.axis("off")
    plt.tight_layout(); plt.show()

## Next steps

- This writes `models/front_end_fold{FOLD}.pth`. Next, run `03_decoder_pipeline.ipynb` (with
  `FREEZE_FRONTEND=True`) for `MODEL="unet"` then `MODEL="vnet"` — both load and freeze this
  front-end, so the decoder is the only difference.
- Per-bone GT is built by `00_gt_per_bone.ipynb`; if you rebuild it, delete the `gt_per_bone_occ_*`
  cache folder so this notebook re-reads the fresh masks.
- If `256^3` runs out of GPU memory: keep `BATCH_SIZE = 1` and ensure `USE_AMP` and `USE_GRAD_CKPT`
  are `True` (the frozen encoder already cuts activation memory).

## 8. Fold x arm sweep driver (C-6 gate matrix)

Runs the **full step-2 pipeline** (train -> save front-end -> gate eval -> per-fold report/diagrams)
for every `(arm, fold)` in **one execution**, so the 5-fold x 2-arm gate matrix is produced without
manually editing `STEP2_TARGET`/`FOLD` and restarting the kernel between runs.

**How to use:** Run All the cells above once (they define every building block; the single-fold cells
in sections 5-7 are harmless - the driver overwrites their fold artifacts and dedups the gate CSV).
Then run the two cells below. Configure `SWEEP_ARMS` / `SWEEP_FOLDS` in the driver cell.

**Per fold it saves** (under `reports/frontend_gate/<arm>/fold<K>/`): `history.csv`, `curves.png`
(loss + M-11 term split + val Dice), `recon_preview.png` (one held-out knee), `gate_metrics.csv`,
and `summary.json`. A running `reports/frontend_gate/sweep_summary.csv` is updated after every fold,
and each fold's rows are merged into `models/decoders/gate_g4_arm_metrics.csv` for section 5 of
`04_decoder_comparison.ipynb`. Cost is ~ (arms x folds) full step-2 trainings, so budget accordingly.

In [ ]:
# ============================ Fold x arm sweep driver (C-6) ============================
# Reuses build_paired_index / kfold_split / make_loader / build_model / run_epoch / evaluate /
# eval_loss_terms / gate_eval defined above. Sets the few globals those functions read (FOLD,
# STEP2_TARGET, USE_TSDF, ARM, per-fold paths) per iteration; everything else stays local.
REPORT_ROOT = ROOT / "reports" / "frontend_gate"
GATE_CSV    = MODELS_DIR / "decoders" / "gate_g4_arm_metrics.csv"

SWEEP_ARMS  = ["binary", "tsdf_blend"]      # arms a, b (extend once C-2/C-3 land)
SWEEP_FOLDS = list(range(N_FOLDS))          # 0 .. N_FOLDS-1

# one clean, un-subsampled index shared by every fold; only the split column changes per fold
sweep_index = build_paired_index()
if not INCLUDE_GEOMETRIC:
    sweep_index = sweep_index[~sweep_index.geometric].reset_index(drop=True)
sweep_index = sweep_index[sweep_index.apply(
    lambda r: has_per_bone_gt(r.dataset, r.case, r.side), axis=1)].reset_index(drop=True)
if SMOKE_TEST:                                  # local plumbing test: a few cases/group (HPC sets False)
    keep = sweep_index[sweep_index.variant == "normal"]
    sub = [g[g.case.isin(list(dict.fromkeys(g.case))[:SMOKE_CASES_PER_GROUP])]
           for _, g in keep.groupby("dataset")]
    sweep_index = pd.concat(sub).reset_index(drop=True)
print("sweep index: %d rows | %d knees | arms=%s | folds=%s"
      % (len(sweep_index), sweep_index.groupby(["dataset", "case", "side"]).ngroups,
         SWEEP_ARMS, SWEEP_FOLDS))

def _save_curves(hist_df, path, arm, fold):
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(hist_df.epoch, hist_df.train_loss, label="train")
    ax[0].plot(hist_df.epoch, hist_df.val_loss, label="val")
    if "val_tsdf" in hist_df and hist_df.val_tsdf.notna().any():
        ax[0].plot(hist_df.epoch, hist_df.val_binary, "--", label="val binary term")
        ax[0].plot(hist_df.epoch, hist_df.val_tsdf, "--", label="val tsdf term")
    ax[0].set_title("loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
    ax[1].plot(hist_df.epoch, hist_df.val_dice, label="val Dice")
    ax[1].set_title("val Dice"); ax[1].set_xlabel("epoch"); ax[1].legend()
    fig.suptitle("%s  fold %d" % (arm, fold))
    fig.tight_layout(); fig.savefig(path, dpi=120, bbox_inches="tight"); plt.close(fig)

def _save_recon(model, loader, path, arm, fold):
    if loader is None:
        return
    batch = next(iter(loader))
    with torch.no_grad():
        out, _ = model(batch["ap"].to(DEVICE), batch["lat"].to(DEVICE))
    pred = (torch.sigmoid(out[0].float()).amax(0).cpu().numpy() > 0.5).astype(float)
    gtv = batch["gt"][0].numpy().max(0); m = pred.shape[0] // 2
    fig, ax = plt.subplots(1, 3, figsize=(11, 4))
    ax[0].imshow(batch["ap"][0, 0], cmap="gray"); ax[0].set_title("input AP")
    ax[1].imshow(gtv[m], cmap="gray", origin="lower"); ax[1].set_title("GT union")
    ax[2].imshow(pred[m], cmap="gray", origin="lower"); ax[2].set_title("prediction union")
    for a in ax:
        a.axis("off")
    fig.suptitle("%s fold %d - %s %s %s"
                 % (arm, fold, batch["dataset"][0], batch["case"][0], batch["side"][0]))
    fig.tight_layout(); fig.savefig(path, dpi=120, bbox_inches="tight"); plt.close(fig)

def run_fold_arm(fold, step2_target):
    global FOLD, STEP2_TARGET, USE_TSDF, ARM, FRONTEND_CKPT, CKPT_DIR
    t0 = time.time()
    FOLD = fold; STEP2_TARGET = step2_target; USE_TSDF = (step2_target == "tsdf_blend")
    ARM = "%s|%s|%s" % ("tsdf" if USE_TSDF else "binary",
                        "unfrozen" if not FREEZE_ENCODER else "frozen",
                        "aux" if globals().get("USE_AUX", False) else "noaux")
    FRONTEND_CKPT = MODELS_DIR / ("front_end_fold%d.pth" % FOLD)
    CKPT_DIR = MODELS_DIR / "decoders" / ("neutral_fold%d" % FOLD); CKPT_DIR.mkdir(parents=True, exist_ok=True)
    rdir = REPORT_ROOT / ARM.replace("|", "-") / ("fold%d" % FOLD); rdir.mkdir(parents=True, exist_ok=True)

    # per-fold split (shared with 03 via the same CSV name) + loaders
    idx = sweep_index.copy()
    idx["split"] = kfold_split(idx, n_folds=N_FOLDS, fold=FOLD, seed=SEED)
    split_csv = MODELS_DIR / "decoders" / ("decoder_split_fold%d.csv" % FOLD)
    idx[["dataset", "case", "side", "variant", "split"]].to_csv(split_csv, index=False)
    tr_loader = make_loader(idx[idx.split == "train"], True)
    va_loader = make_loader(idx[(idx.split == "val") & (idx.variant == "normal")], False)
    te_loader = make_loader(idx[(idx.split == "test") & (idx.variant == "normal")], False)

    model = build_model().to(DEVICE)
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(EPOCHS, 1))
    scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and DEVICE.type == "cuda"))

    history, best_dice = [], -1.0
    for epoch in range(EPOCHS):
        tr, steps = run_epoch(model, tr_loader, optimizer, scaler, True)
        if va_loader:
            va, _ = run_epoch(model, va_loader, optimizer, scaler, False)
            _, overall, _ = evaluate(model, va_loader)
            m11b, m11t = (eval_loss_terms(model, va_loader) if USE_TSDF else (float("nan"), float("nan")))
        else:
            va, overall, m11b, m11t = float("nan"), {"dice": float("nan"), "iou": float("nan")}, float("nan"), float("nan")
        if steps > 0:
            scheduler.step()
        history.append(dict(epoch=epoch, train_loss=tr, val_loss=va, val_dice=overall["dice"],
                            val_iou=overall["iou"], val_binary=m11b, val_tsdf=m11t,
                            lr=optimizer.param_groups[0]["lr"]))
        if overall["dice"] > best_dice:
            best_dice = overall["dice"]
            torch.save({"front_end": model.fusion.state_dict(),
                        "config": {"LIFT_DEPTH": LIFT_DEPTH, "BACKBONE": BACKBONE, "TARGET_RES": TARGET_RES,
                                   "N_CLASSES": N_CLASSES, "BONES": BONES}}, FRONTEND_CKPT)
        print("  [%s fold %d] epoch %03d | train %.4f | val %.4f | val_dice %.4f%s"
              % (ARM, FOLD, epoch, tr, va, overall["dice"],
                 ("" if not USE_TSDF else " | M-11 bin %.4f tsdf %.4f" % (m11b, m11t))))

    hist_df = pd.DataFrame(history); hist_df.to_csv(rdir / "history.csv", index=False)
    _save_curves(hist_df, rdir / "curves.png", ARM, FOLD)
    _save_recon(model, te_loader or va_loader, rdir / "recon_preview.png", ARM, FOLD)

    # gate metrics for this fold (gate_eval reads ARM/FOLD/STEP2_TARGET/FREEZE_ENCODER globals set above)
    gate_loader = te_loader or va_loader
    frac = heal = float("nan"); n_knees = 0
    if gate_loader:
        gdf = gate_eval(model, gate_loader)
        gdf.to_csv(rdir / "gate_metrics.csv", index=False)          # per-fold bundle copy
        if GATE_CSV.exists():                                        # merge into global matrix, dedup (arm,fold)
            prev = pd.read_csv(GATE_CSV); prev = prev[~((prev.arm == ARM) & (prev.fold == FOLD))]
            merged = pd.concat([prev, gdf], ignore_index=True)
        else:
            merged = gdf
        merged.to_csv(GATE_CSV, index=False)
        n_knees = len(gdf)
        frac = gdf[gdf.dataset == "fractured"]["assd_mm"].mean()
        heal = gdf[gdf.dataset == "healthy"]["assd_mm"].mean()

    secs = round(time.time() - t0, 1)
    summary = dict(arm=ARM, fold=FOLD, epochs=EPOCHS, best_val_dice=round(best_dice, 4),
                   n_test_knees=n_knees, frac_assd_mm=round(float(frac), 3),
                   heal_assd_mm=round(float(heal), 3), secs=secs, report_dir=str(rdir))
    with open(rdir / "summary.json", "w") as f:
        json.dump(summary, f, indent=2)
    print("  -> done %s fold %d in %.1fs | best_val_dice %.4f | %d test knees | report %s"
          % (ARM, FOLD, secs, best_dice, n_knees, rdir))
    return summary

In [ ]:
# ---- run the sweep: arms x folds, with live progress + a running summary ----
REPORT_ROOT.mkdir(parents=True, exist_ok=True)
sweep_rows, total, done, sweep_t0 = [], len(SWEEP_ARMS) * len(SWEEP_FOLDS), 0, time.time()
for step2_target in SWEEP_ARMS:
    for fold in SWEEP_FOLDS:
        done += 1
        print("\n===== [%d/%d] arm=%s fold=%d =====" % (done, total, step2_target, fold))
        sweep_rows.append(run_fold_arm(fold, step2_target))
        sw = pd.DataFrame(sweep_rows)
        sw.to_csv(REPORT_ROOT / "sweep_summary.csv", index=False)   # updated after every fold
        print("---- progress %d/%d | elapsed %.1f min ----" % (done, total, (time.time() - sweep_t0) / 60.0))
        print(sw[["arm", "fold", "best_val_dice", "n_test_knees", "frac_assd_mm", "heal_assd_mm", "secs"]].to_string(index=False))

print("\n===== SWEEP COMPLETE =====")
print("per-fold reports/diagrams ->", REPORT_ROOT)
print("gate matrix ->", GATE_CSV, "(run 04 section 5 for the SHIP/hold verdict)")
print(pd.DataFrame(sweep_rows).to_string(index=False))